# Repeat Purchase Prediction

**Note on framing:** an earlier approach attempted to predict "churn" using
days-since-last-purchase. That was abandoned because 97% of Olist customers
purchase only once, making days-since-last-purchase almost entirely a function
of *when* a customer joined rather than their behavior — the label was
confounded with time, not genuinely predictive.

This notebook instead predicts: **will a first-time buyer make a second
purchase?** — using only information available at the time of their first
order, avoiding that confound.

In [1]:
import pandas as pd
import numpy as np

transactions = pd.read_csv('../data/processed/transactions.csv', parse_dates=['order_purchase_timestamp'])

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

print(transactions.shape, orders.shape)

(96477, 4) (99441, 8)


In [2]:
first_order_ids = (
    transactions.sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id')
    .first()
    .reset_index()
    .rename(columns={'order_value': 'first_order_value'})
)[['customer_unique_id', 'order_id', 'first_order_value']]

print(first_order_ids.shape)
first_order_ids.head()

(93357, 3)


,customer_unique_id,order_id,first_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,27.19
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,43.62
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,196.89


In [3]:
customer_frequency = transactions.groupby('customer_unique_id')['order_id'].count().reset_index()
customer_frequency.columns = ['customer_unique_id', 'frequency']

model_df = first_order_ids.merge(customer_frequency, on='customer_unique_id', how='left')
model_df['will_repeat'] = (model_df['frequency'] > 1).astype(int)

# frequency is the direct source of the label — drop it so it isn't used as a feature
model_df = model_df.drop(columns=['frequency'])

print(model_df['will_repeat'].value_counts(normalize=True))

will_repeat
0    0.969997
1    0.030003
Name: proportion, dtype: float64


In [4]:
item_products = (
    items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(orders[['order_id', 'customer_id']], on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
)

# first order ONLY — avoids leaking multi-order category diversity
first_order_items = item_products[item_products['order_id'].isin(first_order_ids['order_id'])]

first_order_diversity = (
    first_order_items.groupby('customer_unique_id')['product_category_name']
    .nunique()
    .reset_index()
    .rename(columns={'product_category_name': 'first_order_category_diversity'})
)

model_df = model_df.merge(first_order_diversity, on='customer_unique_id', how='left')
model_df['first_order_category_diversity'] = model_df['first_order_category_diversity'].fillna(0)

In [5]:
order_payment_type = payments.groupby('order_id')['payment_type'].first().reset_index()

model_df = model_df.merge(
    first_order_ids[['customer_unique_id', 'order_id']]
        .merge(order_payment_type, on='order_id', how='left')[['customer_unique_id', 'payment_type']],
    on='customer_unique_id', how='left'
)
model_df = pd.get_dummies(model_df, columns=['payment_type'], drop_first=True)

In [6]:
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

review_scores = order_reviews.groupby('order_id')['review_score'].mean().reset_index()

first_order_extra = (
    first_order_ids[['customer_unique_id', 'order_id']]
    .merge(orders[['order_id', 'delivery_days', 'delivery_delay_days']], on='order_id', how='left')
    .merge(review_scores, on='order_id', how='left')
)

model_df = model_df.merge(
    first_order_extra[['customer_unique_id', 'delivery_days', 'delivery_delay_days', 'review_score']],
    on='customer_unique_id', how='left'
)

for col in ['delivery_days', 'delivery_delay_days', 'review_score']:
    model_df[col] = model_df[col].fillna(model_df[col].median())

model_df.describe()

,first_order_value,will_repeat,first_order_category_diversity,delivery_days,delivery_delay_days,review_score
count,93357.000000,93357.000000,93357.000000,93357.000000,93357.000000,93357.000000
mean,204.796912,0.030003,0.993541,12.103506,-11.853391,4.159110
std,626.675160,0.170597,0.146188,9.583063,10.184138,1.282776
min,9.590000,0.000000,0.000000,0.000000,-147.000000,1.000000
25%,62.690000,0.000000,1.000000,6.000000,-17.000000,4.000000
50%,109.680000,0.000000,1.000000,10.000000,-12.000000,5.000000
75%,195.350000,0.000000,1.000000,15.000000,-7.000000,5.000000
max,109312.640000,1.000000,3.000000,209.000000,188.000000,5.000000


In [7]:
print(model_df.shape)
print(model_df.isna().sum())
model_df.head()

(93357, 11)
customer_unique_id                0
order_id                          0
first_order_value                 0
will_repeat                       0
first_order_category_diversity    0
payment_type_credit_card          0
payment_type_debit_card           0
payment_type_voucher              0
delivery_days                     0
delivery_delay_days               0
review_score                      0
dtype: int64


,customer_unique_id,order_id,first_order_value,will_repeat,first_order_category_diversity,payment_type_credit_card,payment_type_debit_card,payment_type_voucher,delivery_days,delivery_delay_days,review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,141.90,0,1,True,False,False,6.0,-5.0,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,27.19,0,1,True,False,False,3.0,-5.0,4.0
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,86.22,0,1,True,False,False,25.0,-2.0,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,43.62,0,1,True,False,False,20.0,-12.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,196.89,0,1,True,False,False,13.0,-8.0,5.0


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols = [
    'first_order_value', 'first_order_category_diversity',
    'payment_type_credit_card', 'payment_type_debit_card', 'payment_type_voucher',
    'delivery_days', 'delivery_delay_days', 'review_score'
]

X = model_df[feature_cols].fillna(0)
y = model_df['will_repeat']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)

proba = log_reg.predict_proba(X_test_scaled)[:, 1]
print("Logistic Regression — ROC-AUC:", roc_auc_score(y_test, proba))
print("Logistic Regression — PR-AUC:", average_precision_score(y_test, proba))

Logistic Regression — ROC-AUC: 0.5352417793254669
Logistic Regression — PR-AUC: 0.033592279909929586


In [9]:
coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': log_reg.coef_[0]}).sort_values('coefficient', ascending=False)
coef_df

,feature,coefficient
5,delivery_days,0.068769
1,first_order_category_diversity,0.050632
7,review_score,0.032424
4,payment_type_voucher,0.029179
0,first_order_value,0.007240
2,payment_type_credit_card,0.003712
3,payment_type_debit_card,-0.019456
6,delivery_delay_days,-0.126794


In [10]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, eval_metric='auc')
xgb.fit(X_train, y_train)

xgb_proba = xgb.predict_proba(X_test)[:, 1]
print("XGBoost — ROC-AUC:", roc_auc_score(y_test, xgb_proba))
print("XGBoost — PR-AUC:", average_precision_score(y_test, xgb_proba))

XGBoost — ROC-AUC: 0.5391038597141595
XGBoost — PR-AUC: 0.0339062986428407


In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, eval_metric='auc')
xgb.fit(X_train, y_train)

xgb_proba = xgb.predict_proba(X_test)[:, 1]
print("XGBoost — ROC-AUC:", roc_auc_score(y_test, xgb_proba))
print("XGBoost — PR-AUC:", average_precision_score(y_test, xgb_proba))

XGBoost — ROC-AUC: 0.5391038597141595
XGBoost — PR-AUC: 0.0339062986428407


In [12]:
model_df['repeat_purchase_probability'] = log_reg.predict_proba(scaler.transform(model_df[feature_cols].fillna(0)))[:, 1]
model_df.to_csv('../data/processed/repeat_purchase_model.csv', index=False)
model_df.head()

,customer_unique_id,order_id,first_order_value,will_repeat,first_order_category_diversity,payment_type_credit_card,payment_type_debit_card,payment_type_voucher,delivery_days,delivery_delay_days,review_score,repeat_purchase_probability
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,141.90,0,1,True,False,False,6.0,-5.0,5.0,0.471419
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,27.19,0,1,True,False,False,3.0,-5.0,4.0,0.459387
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,86.22,0,1,True,False,False,25.0,-2.0,3.0,0.483357
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,43.62,0,1,True,False,False,20.0,-12.0,4.0,0.511677
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,196.89,0,1,True,False,False,13.0,-8.0,5.0,0.493487
